In [ ]:
import polars as pl
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import colorir as cl
import numpy as np
from pathlib import Path
import analyses.calculate_adh as adh
from analyses import io
import analyses.phylogeny as phy
from tqdm import tqdm

In [ ]:
celldf = (
    io
        .read_celldfs("../runs/evo/", levels=["replica"])
        .with_columns(
            replica=pl.col("replica").cast(pl.UInt32)
        )
        .sort(["replica", "wtime"])
)
celldf

In [ ]:
selectors = ["index", "ligands", "receptors"]
gammadfs = []
groupped = (
    celldf
        .select(selectors + ["replica", "wtime"])
        .filter(pl.col("wtime") % 2e6 == 0)  # Values only change between seasons of 2e6 steps
        .group_by(["replica", "wtime"])
)
for (replica, time), group in tqdm(groupped):
    sel_group = group.select(selectors)
    lig_recep = sel_group.join(sel_group, how="cross")
    gammas = adh.calculate_gamma(
        20, 
        adh.cell_contact_energy(
            *lig_recep.select(pl.exclude("index", "index_right")).to_numpy().T,
            20,
            32,
            8
        )
    )
    means = lig_recep.with_columns(gamma=gammas).group_by("index").agg(
        pl.col("gamma").mean()
    )
    max_gamma = means[means["gamma"].arg_max(), "index"]
    summary = (
        pl.DataFrame(gammas)
            .describe()
            .with_columns(replica=replica, wtime=time, max_gamma_ix=max_gamma)
            .pivot(on="statistic", values="column_0")
    ).with_columns(sample=np.expand_dims(np.random.choice(gammas, 100), 0))
    gammadfs.append(summary)
gammadf = pl.concat(gammadfs).sort("replica", "wtime")
gammadf
    

In [ ]:
fig = make_subplots(1, 2, column_widths=[0.85, 0.15], shared_yaxes=True, horizontal_spacing=0.01)
colors = cl.StackPalette.load("carnival").resize(gammadf["replica"].n_unique(), repeat=True)
for (replica,), group in gammadf.sort("wtime").group_by("replica", maintain_order=True):
    color = colors[7]
    line = go.Scatter(
        x=group["wtime"],
        y=group["mean"],
        # Lines or markers
        mode="lines",
        line_color=color,
        line_width=6,
        opacity=0.1,
        legendgroup=replica,
        name=replica,
        marker_size=8,
        showlegend=False
    )
    fig.add_trace(line, row=1, col=1)
fig.add_trace(go.Histogram(
    y=gammadf.filter(wtime=pl.col("wtime").max())["mean"],
    nbinsy=12,
    marker_color="#999999",
    showlegend=False
), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    xaxis_title="time",
    yaxis_title="mean gamma",
    width=700,
    height=400,
    bargap=0.1
)
fig.update_xaxes(
    showgrid=False,
    showticklabels=False,
    col=2
)
fig.update_yaxes(
    showgrid=False,
    col=2
)
fig.update_xaxes(tickangle=0)
io.save_plot(fig, "../plots/adhesion_evolution")

In [ ]:
sampledf = gammadf.filter(pl.col("wtime") > 150e6).group_by("replica").agg(sample=pl.concat_arr(pl.col("sample")))
sampledf

In [ ]:
ancdf = phy.matrix(celldf.filter(replica=0), 2e6)
ancdf

In [ ]:
from ete3 import Tree

prev_time = "0"
nodes = [None] * ancdf[prev_time].n_unique()
for x in ancdf[prev_time].unique():
    t = Tree(name=str(x))
    t.add_feature("time", "0")
    nodes[x] = t
for row in ancdf.transpose():
    row = row[::-1]
    prev_anc = row[0]
    node = nodes[prev_anc]
    for anc in row.filter(row.is_not_null())[1:]:
        anc_str = str(anc)
        found = False
        for child in node.children:
            if child.name == anc_str:
                found = True
                break
        if found:
            node = child
        else:
            time = str(2_000_000 + int(node.time))
            node = node.add_child(name=anc_str)
            node.add_feature("time", time)
print(nodes[5])

In [ ]:
ancmelt = (
    ancdf
        # "lineage" is a unique identifier for leaf and corresponding ancestry path (both extinct and non-extinct)
        # "og" is the cell id of each lineage at the root (wtime == 0)
        # "id" is the cell id of the lineage at that time in the simulation
        .with_columns(lineage=np.arange(ancdf.height), og=ancdf["0"])
        .unpivot(index=["lineage", "og"], value_name="index", variable_name="wtime")
        .with_columns(pl.col("wtime").cast(int))
        .sort(["lineage", "wtime"])
)
ancmelt


In [ ]:
fitdf = ancmelt.join(
    gammadf.filter(replica=0).with_columns(index="max_gamma_ix"), 
    left_on=["wtime", "index"], 
    right_on=["wtime", "max_gamma_ix"]
).select(
    "lineage",
    "index",
    next_wtime=pl.col("wtime") + 2_000_000
).join(
    ancmelt.select("wtime", "lineage", "index"),
    left_on=["next_wtime", "lineage"],
    right_on=["wtime", "lineage"]
)
fitdf

In [ ]:
ndescdf = fitdf.group_by("index", "next_wtime").agg(
    # Nulls represent that the lineage where index == index_right died
    # Null count should return either 0 or 1
    n_desc=pl.col("index_right").n_unique() - (pl.col("index_right").null_count() == 1),
).with_columns()
ndescdf

In [ ]:
px.histogram(ndescdf, x="n_desc")

In [ ]:
px.line(ancmelt, x="index", y="wtime", color="lineage").update_traces(visible="legendonly")

In [ ]:
descdf = ancmelt.group_by(["og", "wtime"]).agg(n_descendents=pl.col("index").drop_nulls().n_unique()).sort("og", "wtime")
descdf

In [ ]:
# First common ancestor of whole pop at wtime = 58e6
px.bar(
    descdf,
    x="wtime", 
    y="n_descendents", 
    color="og",
    barmode="stack"
).update_traces(
    marker_line_color="rgba(0, 0, 0, 0)"
).update_layout(
    bargap=0
)